# Re-annoter le 800k (externe) et le 110k avec le prof bidoua-aware -- via rclone (sans drive.mount)

**Pourquoi rclone et pas `drive.mount()`** : le flux d'authentification
`dfs_ephemeral` de Colab echoue (400 `credentials-propagation`) sur ce
compte. On contourne en accedant a Drive via **rclone** avec le meme
token OAuth deja configure en local pour ce projet -- independant du flux
casse de Colab.

**Plusieurs sessions/comptes en parallele (reservation de shard)** : le
tier gratuit de Colab alloue ~2 vCPU/session -- a ce debit, le volume
complet (110k + 800k) prendrait des mois en une seule session. Comme pour
le dataset externe original (`dataset_full_matrix_colab_1..5`), ce MEME
notebook peut tourner dans **plusieurs sessions independantes** (tes
propres onglets, ou d'autres comptes Google invites sur le Drive
partage). Chaque session **reserve automatiquement** un shard libre au
demarrage (petit fichier JSON partage sur Drive, cf. cellule "Reservation"
juste avant chaque section) -- pas de coordination manuelle a faire,
pas deux sessions sur les memes positions. Le nombre de shards
(`NUM_SHARDS_*`) est volontairement plus fin que le nombre de sessions
attendu : une session qui termine son shard peut relancer la cellule de
reservation pour en prendre un autre, ca s'auto-equilibre.

**Important -- ce n'est PAS une vraie base concurrente** : le fichier de
reservation est lu-puis-ecrit via `rclone` (pas de verrou reseau reel).
Sur une poignee de sessions demarrees manuellement par des humains, le
risque de collision (deux sessions qui lisent le fichier au meme
instant exact) est negligeable en pratique -- mais ce n'est pas garanti
a 100%. Le cache SQLite, lui, NE PEUT PAS servir d'etat partage en temps
reel entre plusieurs machines (pas de verrou a travers une synchro
Drive) -- chaque shard a son propre cache, jamais partage entre sessions.

Principe general :
1. rclone est configure sur Colab a partir de TON token (colle via un
   champ masque, jamais ecrit en clair dans le notebook).
2. Les datasets d'entree sont **telecharges** depuis Drive vers le disque
   local Colab (rapide, partages entre shards).
3. Chaque section (A = externe, B = 110k) reserve un shard libre au
   demarrage, puis l'annotation tourne sur le disque local Colab. Le
   cache SQLite + les sorties sont **resynchronises periodiquement vers
   Drive** en tache de fond (pour survivre aux deconnexions).
4. En cas de deconnexion : re-executer les cellules dans l'ordre -- la
   reservation retrouve le MEME shard (deja a ton nom), le cache reprend
   exactement ou il s'etait arrete.

**ATTENTION CACHE (piege reel)** : le cache d'annotations est indexe par
position + config du professeur, PAS par version du code d'evaluation.
Tous les dossiers de cache portent un suffixe `_bidoua_v1` : ne jamais
les faire pointer vers un ancien cache (pre-correctif).

**Debit a mesurer sur CE Colab avant le volume complet** (ne pas
supposer) : ~15-16s/position mesure autrefois sur l'externe, ~61s/position
mesure en local sur le 110k (budget de 60s presque toujours epuise). Deux
chiffres non interchangeables -> toujours piloter d'abord.

**Pour inviter d'autres comptes** : partage le dossier Drive `SongoFish`
(Editeur) avec chaque compte invite, donne-leur ce notebook (branche
`dev` sur GitHub -- il faudra aussi les ajouter comme collaborateurs du
depot prive, ou leur donner un token GitHub en lecture seule) et ton
token rclone RESTE LE TIEN, chacun doit generer/utiliser SON PROPRE
acces (soit son propre token rclone vers le Drive partage, soit -- plus
simple pour des invites -- monter le Drive partage differemment). A
decider selon qui rejoint concretement.


## 1 -- Configurer rclone : saisir le token dans le champ masque

**Recuperer le token** : sur ton Mac (avec le NOUVEAU token, apres
revocation + `rclone config reconnect songo_drive:`) :
```bash
python3 -c "import configparser,os; c=configparser.ConfigParser(); c.read(os.path.expanduser('~/.config/rclone/rclone.conf')); print(c['songo_drive']['token'])"
```
Ca affiche une ligne qui commence par `{"access_token":...`.

**IMPORTANT -- ne PAS coller le token dans le code de la cellule** (c'est
ce qui l'avait expose la derniere fois). Execute la cellule telle quelle :
un **champ de saisie masque apparait** (en haut de la fenetre dans VS
Code, ou juste sous la cellule sur le site Colab). Colle le token
DEDANS, puis Entree.

In [18]:
import os, subprocess, getpass

# Installer le binaire rclone sur Colab
subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True, check=False)

# >>> NE RIEN MODIFIER SUR CETTE LIGNE <<< : execute la cellule, puis colle
# le token dans le champ masque qui apparait (PAS dans le code ci-dessous).
token = getpass.getpass("Colle le token rclone ici puis Entree : ").strip()

assert token.startswith("{") and "access_token" in token, \
    "Token invalide : doit commencer par { et contenir access_token (colle-le dans le CHAMP, pas dans le code)"

os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[songo_drive]\ntype = drive\nscope = drive\ntoken = %s\n" % token)
os.chmod("/root/.config/rclone/rclone.conf", 0o600)
del token  # ne pas garder en memoire

# Test : doit lister le contenu de SongoFish/data sur ton Drive
print(subprocess.run("rclone lsd songo_drive:SongoFish/data", shell=True,
                     capture_output=True, text=True).stdout)

           0 2026-07-20 17:48:40        -1 .tmp.driveupload
           0 2026-07-20 12:53:34        -1 checkpoints
           0 2026-07-19 19:34:43        -1 dataset_curated_endgames
           0 2026-07-20 12:39:41        -1 dataset_curated_endgames (1)
           0 2026-07-20 12:54:04        -1 dataset_curated_endgames (1) (1)
           0 2026-07-20 12:54:05        -1 dataset_curated_endgames (1) (1) (1)
           0 2026-07-20 12:54:04        -1 dataset_full_matrix_merged_all_colabs
           0 2026-07-20 12:54:05        -1 dataset_v001_10k
           0 2026-07-16 17:39:25        -1 dataset_v002_100k
           0 2026-07-20 12:39:54        -1 dataset_v002_100k (1)
           0 2026-07-20 12:54:05        -1 dataset_v002_100k (1) (1)
           0 2026-07-20 12:56:51        -1 dataset_v002_100k (1) (1) (1)
           0 2026-07-16 21:01:37        -1 dataset_v003_110k
           0 2026-07-20 12:39:41        -1 dataset_v003_110k (1)
           0 2026-07-20 12:54:05        -1 dataset_v00

## 2 -- Cloner le depot (prive) et installer

Le depot `GlennEriss/songo-fish` est **prive** -> le clone exige un token
GitHub (un `git clone` anonyme echoue avec `could not read Username`).

**Creer un token GitHub (fine-grained, lecture seule sur ce repo)** :
1. GitHub -> Settings -> Developer settings -> **Personal access tokens**
   -> **Fine-grained tokens** -> *Generate new token*.
2. **Repository access** : *Only select repositories* -> `songo-fish`.
3. **Permissions** -> *Repository permissions* -> **Contents : Read-only**.
4. Generer, copier le token (commence par `github_pat_...`).

Execute la cellule ci-dessous : colle ce token dans le **champ masque**
qui apparait (PAS dans le code). Le token est retire de la config git
juste apres le clone, il ne reste pas stocke dans la session.

In [25]:
import getpass, subprocess, os

REPO_DIR = "/content/songo-fish"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Deja clone lors d'une execution precedente de cette session -- ne
    # pas re-cloner (git refuse dans un dossier non vide, provoquait
    # auparavant un faux message "verifie le token" alors que le vrai
    # depot etait deja la). On se contente de se mettre a jour.
    print(f"{REPO_DIR} existe deja, mise a jour (fetch + checkout dev) au lieu d'un nouveau clone.")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "dev"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "dev"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/dev"], check=True)
    print("A jour (branche dev).")
else:
    # >>> NE RIEN MODIFIER ICI <<< : execute la cellule, colle le token GitHub
    # dans le champ masque qui apparait (PAS dans le code).
    gh_token = getpass.getpass("Colle ton token GitHub (github_pat_...) puis Entree : ").strip()
    assert gh_token, "token vide"

    _url = f"https://{gh_token}@github.com/GlennEriss/songo-fish.git"
    res = subprocess.run(["git", "clone", "-b", "dev", _url, REPO_DIR],
                         capture_output=True, text=True)
    del gh_token, _url  # ne pas garder le token en memoire
    if res.returncode != 0:
        # Afficher la VRAIE raison de git (jamais avalee) -- peut etre le
        # token (expire/mauvaise permission), un probleme reseau, etc.
        print("--- stderr git ---")
        print(res.stderr)
        raise SystemExit("Echec du clone -- voir le message git ci-dessus pour la vraie cause.")

    # Retirer toute trace du token : remettre l'URL du remote sans identifiant.
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    "https://github.com/GlennEriss/songo-fish.git"], check=True)
    print("Clone OK (branche dev).")


/content/songo-fish existe deja, mise a jour (fetch + checkout dev) au lieu d'un nouveau clone.


CalledProcessError: Command '['git', '-C', '/content/songo-fish', 'fetch', 'origin', 'dev']' returned non-zero exit status 128.

In [26]:
%cd /content/songo-fish
!pip install -q -e ".[train]"


/content/songo-fish
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for songo-ai (pyproject.toml) ... done


In [27]:
import multiprocessing
NUM_WORKERS = multiprocessing.cpu_count()
print('coeurs disponibles sur cette session Colab:', NUM_WORKERS)


coeurs disponibles sur cette session Colab: 2


## 3 -- Identifiant de session + chemins communs + telechargement + synchro

`WHO` sert a la reservation de shard (cellules suivantes, une par
section) : mets ton email ou un pseudo qui t'identifie -- si cette
session se reconnecte plus tard avec le MEME `WHO`, elle retrouve
automatiquement le shard qu'elle avait deja reserve, pas un nouveau.

In [28]:
import subprocess, os, signal, json as _json

WHO = input("Ton identifiant pour la reservation de shard (email ou pseudo) : ").strip()
assert WHO, "identifiant vide"

# --- Distant (Drive, via rclone) / Local (disque Colab, ephemere) ---
REMOTE = "songo_drive:SongoFish/data"
L = "/content/songo_data"
L_EXTERNAL_IN = f"{L}/dataset_full_matrix_merged_all_colabs"
L_OWN_IN      = f"{L}/dataset_v003_110k"
os.makedirs(L, exist_ok=True)

def rc(cmd):
    print(">", cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

# Telecharger les entrees Drive -> local (rapide, une fois -- partagees entre TOUS les shards/sessions)
rc(f'rclone copy "{REMOTE}/dataset_full_matrix_merged_all_colabs" "{L_EXTERNAL_IN}" --progress')
rc(f'rclone copy "{REMOTE}/dataset_v003_110k" "{L_OWN_IN}" --progress')

# --- Synchro periodique local -> Drive, en tache de fond ---
_sync_proc = None
def start_bg_sync(local_dir, remote_dir, every_s=300):
    global _sync_proc
    stop_bg_sync()
    script = (f'while true; do sleep {every_s}; '
              f'rclone copy "{local_dir}" "{remote_dir}" >/dev/null 2>&1; done')
    _sync_proc = subprocess.Popen(["bash", "-c", script], preexec_fn=os.setsid)
    print(f"[sync] tache de fond demarree : {local_dir} -> {remote_dir} toutes les {every_s}s")

def stop_bg_sync():
    global _sync_proc
    if _sync_proc is not None:
        os.killpg(os.getpgid(_sync_proc.pid), signal.SIGTERM)
        _sync_proc = None

def push(local_dir, remote_dir):
    rc(f'rclone copy "{local_dir}" "{remote_dir}"')
    print("[sync] push final termine")

# --- Reservation de shard (fichier JSON partage sur Drive, PAS une vraie
# base concurrente -- lu-puis-ecrit via rclone, suffisant pour une
# poignee de sessions demarrees manuellement, cf. cellule 0) ---
CLAIMS_REMOTE = f"{REMOTE}/_bidoua_shard_claims.json"
CLAIMS_LOCAL = "/content/_bidoua_shard_claims.json"

def _load_claims() -> dict:
    r = subprocess.run(f'rclone cat "{CLAIMS_REMOTE}"', shell=True, capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        return {}
    try:
        return _json.loads(r.stdout)
    except Exception:
        return {}

def _save_claims(claims: dict) -> None:
    with open(CLAIMS_LOCAL, "w") as f:
        _json.dump(claims, f, indent=2)
    rc(f'rclone copyto "{CLAIMS_LOCAL}" "{CLAIMS_REMOTE}"')

def claim_shard(base_name: str, num_shards: int) -> int:
    """Reserve le premier shard libre de `base_name` (parmi 0..num_shards-1)
    pour WHO, ou retrouve le shard deja reserve par WHO si deja fait (reprise
    apres deconnexion). Leve une erreur si tout est deja pris."""
    key = f"{base_name}_of{num_shards}"
    claims = _load_claims()
    entry = claims.get(key, {})
    for idx_str, who in entry.items():
        if who == WHO:
            print(f"[reservation] reprise du shard {idx_str} deja reserve par {WHO} ({key})")
            return int(idx_str)
    taken = {int(i) for i in entry.keys()}
    free = [i for i in range(num_shards) if i not in taken]
    if not free:
        raise RuntimeError(f"Tous les shards de {key} sont deja reserves ({len(taken)}/{num_shards}) -- augmenter num_shards ou attendre qu'un shard se libere (pas gere automatiquement).")
    chosen = free[0]
    entry[str(chosen)] = WHO
    claims[key] = entry
    _save_claims(claims)
    print(f"[reservation] shard {chosen}/{num_shards} reserve pour {WHO} ({key})")
    return chosen


## Section A -- dataset externe (883k, vraies parties)

### A.1 -- pilote (mesurer le debit reel sur CE Colab)

In [ ]:
# >>> A adapter si besoin : nombre TOTAL de shards pour le dataset externe.
# Plus fin que le nombre de sessions attendu -- une session qui termine
# son shard peut relancer cette cellule pour en reserver un autre.
NUM_SHARDS_EXTERNAL = 40

SHARD_INDEX_EXTERNAL = claim_shard("dataset_external_reannotated_bidoua_v1", NUM_SHARDS_EXTERNAL)
SHARD_SUFFIX = f"_shard{SHARD_INDEX_EXTERNAL}of{NUM_SHARDS_EXTERNAL}"

R_EXTERNAL_OUT = f"{REMOTE}/dataset_external_reannotated_bidoua_v1{SHARD_SUFFIX}"
L_EXTERNAL_OUT = f"{L}/dataset_external_reannotated_bidoua_v1{SHARD_SUFFIX}"
L_EXTERNAL_CACHE = f"{L_EXTERNAL_OUT}/annotation_cache"
os.makedirs(L_EXTERNAL_CACHE, exist_ok=True)

# Reprise : re-tirer le cache + sorties DEJA constitues sur Drive pour CE shard (no-op au 1er lancement)
rc(f'rclone copy "{R_EXTERNAL_OUT}" "{L_EXTERNAL_OUT}"')


In [ ]:
!python3 apps/trainer/scripts/reannotate_external_dataset.py \
    --input-dir "{L_EXTERNAL_IN}" \
    --out-dir /content/pilot_external_bidoua \
    --cache-dir "{L_EXTERNAL_CACHE}" \
    --shard-index {SHARD_INDEX_EXTERNAL} --num-shards {NUM_SHARDS_EXTERNAL} \
    --limit 100 --num-workers {NUM_WORKERS}


### A.2 -- volume complet (une fois le debit du pilote juge acceptable)

La resync vers Drive tourne en fond (toutes les 5 min) + un push final.
Le cache etant sauvegarde, une deconnexion se reprend en re-executant les
cellules dans l'ordre (l'etape 3 re-tire le cache, l'annotation saute le
deja-fait).

In [ ]:
start_bg_sync(L_EXTERNAL_OUT, R_EXTERNAL_OUT, every_s=300)
try:
    !python3 apps/trainer/scripts/reannotate_external_dataset.py \
        --input-dir "{L_EXTERNAL_IN}" \
        --out-dir "{L_EXTERNAL_OUT}" \
        --cache-dir "{L_EXTERNAL_CACHE}" \
        --shard-index {SHARD_INDEX_EXTERNAL} --num-shards {NUM_SHARDS_EXTERNAL} \
        --num-workers {NUM_WORKERS}
finally:
    stop_bg_sync()
    push(L_EXTERNAL_OUT, R_EXTERNAL_OUT)


## Section B -- notre dataset (110k, deja au format interne)

### B.1 -- pilote

In [ ]:
# >>> A adapter si besoin : nombre TOTAL de shards pour le dataset 110k.
NUM_SHARDS_OWN = 20

SHARD_INDEX_OWN = claim_shard("dataset_v003_110k_bidoua_v1", NUM_SHARDS_OWN)
SHARD_SUFFIX_OWN = f"_shard{SHARD_INDEX_OWN}of{NUM_SHARDS_OWN}"

R_OWN_OUT = f"{REMOTE}/dataset_v003_110k_bidoua_v1{SHARD_SUFFIX_OWN}"
L_OWN_OUT = f"{L}/dataset_v003_110k_bidoua_v1{SHARD_SUFFIX_OWN}"
L_OWN_CACHE = f"{L_OWN_OUT}/annotation_cache"
os.makedirs(L_OWN_CACHE, exist_ok=True)

rc(f'rclone copy "{R_OWN_OUT}" "{L_OWN_OUT}"')


In [ ]:
!python3 apps/trainer/scripts/reannotate_own_dataset.py \
    --input-dir "{L_OWN_IN}" \
    --out-dir /content/pilot_110k_bidoua \
    --cache-dir "{L_OWN_CACHE}" \
    --shard-index {SHARD_INDEX_OWN} --num-shards {NUM_SHARDS_OWN} \
    --limit 30 --num-workers {NUM_WORKERS}


### B.2 -- volume complet

In [ ]:
start_bg_sync(L_OWN_OUT, R_OWN_OUT, every_s=300)
try:
    !python3 apps/trainer/scripts/reannotate_own_dataset.py \
        --input-dir "{L_OWN_IN}" \
        --out-dir "{L_OWN_OUT}" \
        --cache-dir "{L_OWN_CACHE}" \
        --shard-index {SHARD_INDEX_OWN} --num-shards {NUM_SHARDS_OWN} \
        --num-workers {NUM_WORKERS}
finally:
    stop_bg_sync()
    push(L_OWN_OUT, R_OWN_OUT)


## Section C -- vraies parties humaines (575 positions)

Positions de VRAIS matchs joueurs (`data/real_matches/match_moves_v1.jsonl`,
versionne -> vient avec le clone du depot, pas besoin de Drive pour
l'entree). Petit volume mais precieux (distribution de jeu reelle). On ne
garde PAS le coup humain comme cible, seulement la position : le prof
l'annote comme les autres datasets.

In [ ]:
# Peu de positions (575) -> peu de shards suffisent.
NUM_SHARDS_REAL = 8

SHARD_INDEX_REAL = claim_shard("dataset_real_matches_bidoua_v1", NUM_SHARDS_REAL)
SHARD_SUFFIX_REAL = f"_shard{SHARD_INDEX_REAL}of{NUM_SHARDS_REAL}"

R_REAL_OUT = f"{REMOTE}/dataset_real_matches_bidoua_v1{SHARD_SUFFIX_REAL}"
L_REAL_OUT = f"{L}/dataset_real_matches_bidoua_v1{SHARD_SUFFIX_REAL}"
L_REAL_CACHE = f"{L_REAL_OUT}/annotation_cache"
REAL_INPUT = "data/real_matches/match_moves_v1.jsonl"  # dans le clone du depot
os.makedirs(L_REAL_CACHE, exist_ok=True)

rc(f'rclone copy "{R_REAL_OUT}" "{L_REAL_OUT}"')  # reprise (no-op au 1er lancement)


### C.1 -- pilote

In [ ]:
!python3 apps/trainer/scripts/annotate_real_matches.py \
    --input "{REAL_INPUT}" \
    --out-dir /content/pilot_real_bidoua \
    --cache-dir "{L_REAL_CACHE}" \
    --shard-index {SHARD_INDEX_REAL} --num-shards {NUM_SHARDS_REAL} \
    --limit 20 --num-workers {NUM_WORKERS}


### C.2 -- volume complet

In [ ]:
start_bg_sync(L_REAL_OUT, R_REAL_OUT, every_s=300)
try:
    !python3 apps/trainer/scripts/annotate_real_matches.py \
        --input "{REAL_INPUT}" \
        --out-dir "{L_REAL_OUT}" \
        --cache-dir "{L_REAL_CACHE}" \
        --shard-index {SHARD_INDEX_REAL} --num-shards {NUM_SHARDS_REAL} \
        --num-workers {NUM_WORKERS}
finally:
    stop_bg_sync()
    push(L_REAL_OUT, R_REAL_OUT)


## Etape finale -- fusion en UN SEUL dataset d'entrainement

A executer UNE SEULE FOIS, une fois que TOUTES les sessions/comptes ont
termine et pousse leurs shards sur Drive. Deux etapes :

1. **Recombiner les shards de chaque source** (externe 800k, 110k, vraies
   parties) -- decouverte automatique des shards presents sur Drive.
2. **Fusionner les trois sources en un seul corpus** (`dataset_v006_bidoua_all`)
   avec deduplication globale : c'est le dataset d'entrainement final, ou
   les vraies parties humaines sont melangees aux 800k/110k re-annotes.

`merge_releases` garde les splits coherents (une position ne change jamais
de train/val/test) et deduplique entre sources (deux sources peuvent
produire la meme position par des chemins differents).

**Pas encore inclus** : `dataset_v004_290k`/`dataset_v005_400k` (pas
encore re-annotes avec le bidoua) -- decision separee, a traiter "en plus"
quand on voudra les ajouter (il suffira de lancer une campagne
reannotate_own_dataset dessus puis de les ajouter a la liste de fusion
ci-dessous).

In [ ]:
# Necessite que rclone soit deja configure (cellule 1) et que les
# sessions dont on veut les resultats aient bien pousse sur Drive.
from pathlib import Path
from songo_ai.dataset import merge_releases

def merge_sharded_dataset(base_name: str):
    """Recombine les shards d'UNE source. Renvoie le chemin local du
    dataset fusionne (ou None si aucun shard trouve)."""
    # lsf --dirs-only : un nom de dossier par ligne, sans colonnes de
    # metadonnees a parser (plus robuste que lsd si un nom contient des
    # espaces, cf. les dossiers "(1)" issus d'un vieux souci de mirroring).
    listing = subprocess.run(f'rclone lsf --dirs-only "{REMOTE}"', shell=True, capture_output=True, text=True).stdout
    shard_names = [name.rstrip("/") for name in listing.splitlines() if f"{base_name}_shard" in name]
    if not shard_names:
        print(f"{base_name}: aucun shard trouve sur Drive, ignore.")
        return None
    print(f"{base_name}: {len(shard_names)} shard(s) trouve(s) : {shard_names}")

    local_root = Path(f"/content/merge_{base_name}")
    shard_dirs = []
    for name in shard_names:
        local_shard = local_root / name
        rc(f'rclone copy "{REMOTE}/{name}" "{local_shard}"')
        shard_dirs.append(local_shard)

    local_out = local_root / "merged"
    manifest = merge_releases(shard_dirs, local_out, deduplicate=True)
    print(f"{base_name}: {manifest['total_positions']} positions, {manifest['counts_per_split']}")
    push(str(local_out), f"{REMOTE}/{base_name}")
    return local_out

# --- Etape 1 : recombiner les shards de chaque source ---
sources = []
for base in ("dataset_external_reannotated_bidoua_v1",
             "dataset_v003_110k_bidoua_v1",
             "dataset_real_matches_bidoua_v1"):
    merged = merge_sharded_dataset(base)
    if merged is not None:
        sources.append(merged)

# --- Etape 2 : fusionner les trois sources en UN corpus d'entrainement ---
assert sources, "aucune source disponible -- rien a fusionner"
FINAL_NAME = "dataset_v006_bidoua_all"
final_local = Path(f"/content/{FINAL_NAME}")
manifest = merge_releases(sources, final_local, deduplicate=True)
print(f"\n=== {FINAL_NAME} : {manifest['total_positions']} positions, {manifest['counts_per_split']} ===")
push(str(final_local), f"{REMOTE}/{FINAL_NAME}")
print(f"Corpus final pousse sur Drive : {REMOTE}/{FINAL_NAME}")
